In [ ]:
import importlib
import json
import os
import re
import subprocess
import sys
import time
import warnings
from types import SimpleNamespace

import numpy as np
import pandas as pd

for _pkg in ('lightgbm', 'catboost', 'optuna'):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])

import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==============================================================================
# [0] KONFIGURASI
# ==============================================================================
MODE = 'full'                # 'smoke' | 'quick' | 'full'
TARGET = 'Sales_per_customer'   # 'Sales_per_customer' (setara jurnal rival) | 'Sales'
SPLIT_MODE = 'temporal'       # 'temporal' (terkelompok) | 'grouped_random' (diagnostik pergeseran temporal)
MAIN_LEVEL = 'strict'          # arena optimasi: 'clean' | 'strict'
MODELS = ['ridge', 'lgbm', 'cat']   # jalankan satu per satu di Colab bila perlu: ['lgbm']
RUN_PIPELINE = True           # False = hanya definisikan fungsi/variabel (untuk diagnosis manual), main() tidak jalan
ONLY_TABLE1 = False           # True = berhenti setelah Tabel 1 (cepat); cocok untuk uji SPLIT_MODE
RUN_RIVAL = True              # replikasi protokol rival (Tabel 7); dengan ONLY_TABLE1=True cocok untuk run grouped_random
RUN_LR_SWEEP = True          # sweep learning rate fixed-300 vs early-stopping (lambat; aktifkan di run final)
LR_SWEEP_VALUES = [0.01, 0.05, 0.1, 0.17, 0.2]
SWEEP_LEVELS = ['leak']
BUDGET_S = None               # None = paritas jumlah trial; 600 = paritas waktu 10 menit (klaim jurnal rival)
RUN_PROBE = True              # cetak estimasi runtime berbasis pengukuran

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

RAW_CSV_PATH = os.environ.get('DATACO_CSV', '/content/drive/MyDrive/DataCoSupplyChainDataset.csv')
RESULTS_DIR = os.environ.get('RESULTS_DIR', '/content/drive/MyDrive/run_T_temporal')
os.makedirs(RESULTS_DIR, exist_ok=True)

C_OVER, C_UNDER = 0.10, 0.20            # ASUMSI biaya per unit: over-prediksi, under-prediksi
TAU = C_UNDER / (C_UNDER + C_OVER)      # kuantil optimal = 0.667
SPLIT_Q = (0.65, 0.75, 0.85)   # batas kumulatif: train | val-tune | val-calib | test
CONF_ALPHA = 0.10              # miskoverage interval (cakupan nominal 90%)
RIVAL_PARAMS = dict(iterations=300, depth=8, l2_leaf_reg=5, bagging_temperature=0.2, border_count=64,
                    bootstrap_type='Bayesian')   # bagging_temperature hanya berlaku untuk bootstrap Bayesian
GWO_LAMBDA, MIN_FEATS = 0.01, 3
LGB_MAX_ITER, CAT_MAX_ITER, ES_ROUNDS = 800, 400, 40
RATIOS = [(1, 1), (2, 1), (3, 1), (5, 1)]   # c_under : c_over untuk analisis sensitivitas

MODE_CFG = {
    'smoke': dict(rows=20000, seeds=[0], n_trials=4, wolves=4, iters=3,
                  gwo_tr=4000, gwo_va=2000, cat_tune_n=8000, boot=200),
    'quick': dict(rows=None, seeds=[0], n_trials=15, wolves=6, iters=5,
                  gwo_tr=20000, gwo_va=10000, cat_tune_n=40000, boot=1000),
    'full':  dict(rows=None, seeds=[0, 1, 2], n_trials=40, wolves=10, iters=8,
                  gwo_tr=30000, gwo_va=15000, cat_tune_n=60000, boot=2000),
}
CFG = MODE_CFG[MODE]

ARMS = ['A_default_MSE', 'B_default_cost', 'C_optuna', 'D_gwo', 'E_gwo_optuna', 'F_random']


def san(c):
    return re.sub(r'[^0-9a-zA-Z_]+', '_', str(c)).strip('_')


PII_TEXT = [san(c) for c in ['Product Description', 'Product Image', 'Customer Email', 'Customer Password',
                             'Customer Fname', 'Customer Lname', 'Customer Street', 'Customer Zipcode',
                             'Order Zipcode']]
CAT_NUMERIC = [san(c) for c in ['Category Id', 'Department Id', 'Product Card Id',
                                'Product Category Id', 'Order Item Cardprod Id']]
PRICE_LEAK = [san(c) for c in ['Sales per customer', 'Order Item Total', 'Order Item Product Price',
                               'Product Price', 'Order Item Discount', 'Benefit per order',
                               'Order Profit Per Order']]
ID_COLS = [san(c) for c in ['Order Id', 'Order Item Id', 'Customer Id', 'Order Customer Id']]
POST_ORDER = [san(c) for c in ['Days for shipping (real)', 'Delivery Status', 'Late_delivery_risk',
                               'Order Status', 'Order Item Profit Ratio']] + \
             ['ship_month', 'ship_day', 'ship_weekday', 'ship_hour']
PRODUCT_PROXY = [san(c) for c in ['Product Card Id', 'Product Name', 'Category Id', 'Category Name',
                                  'Product Category Id', 'Order Item Cardprod Id', 'Department Id',
                                  'Department Name', 'Product Status']]
PRICE_LEAK_T = [c for c in PRICE_LEAK if c != TARGET] + ([] if TARGET == 'Sales' else ['Sales'])
LEVEL_DROP = {
    'leak': [],
    'clean': PRICE_LEAK_T + ID_COLS,
    'strict': PRICE_LEAK_T + ID_COLS + POST_ORDER + PRODUCT_PROXY,
}
LEVEL_DROP['strict_nocal'] = LEVEL_DROP['strict'] + ['order_month', 'order_day', 'order_weekday', 'order_hour']   # uji: sinyal L2 dari kalender?

DEFAULTS = {
    'ridge': {'alpha': 1.0},
    'lgbm': {'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 20,
             'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 0.0},
    'cat': {'learning_rate': 0.1, 'depth': 6, 'l2_leaf_reg': 3.0},
}
PROXY = {   # model ringan untuk evaluasi fitness GWO
    'ridge': DEFAULTS['ridge'],
    'lgbm': {**DEFAULTS['lgbm'], 'learning_rate': 0.15, 'max_iter': 60},
    'cat': {**DEFAULTS['cat'], 'learning_rate': 0.2, 'depth': 4, 'max_iter': 60},
}

S = SimpleNamespace()
RIDGE_CACHE = {}
ROW_COSTS = {}
PRED = {}
SELECTED = {}
GWO_CURVES = {}


# ==============================================================================
# [1] DATA: LOAD, FITUR WAKTU, SPLIT TEMPORAL TERKELOMPOK
# ==============================================================================
def load_dataset(cfg):
    print('[*] Memuat dataset ...')
    df = pd.read_csv(RAW_CSV_PATH, encoding='latin-1')
    df.columns = [san(c) for c in df.columns]
    if cfg['rows'] and cfg['rows'] < len(df):
        df = df.sample(cfg['rows'], random_state=0).reset_index(drop=True)
    df = df.drop(columns=[c for c in PII_TEXT if c in df.columns])
    df['order_dt'] = pd.to_datetime(df['order_date_DateOrders'], errors='coerce')
    df['ship_dt'] = pd.to_datetime(df['shipping_date_DateOrders'], errors='coerce')
    df = df[df['order_dt'].notna() & df[TARGET].notna()].reset_index(drop=True)
    for p, col in (('order', 'order_dt'), ('ship', 'ship_dt')):
        df[f'{p}_month'] = df[col].dt.month
        df[f'{p}_day'] = df[col].dt.day
        df[f'{p}_weekday'] = df[col].dt.dayofweek
        df[f'{p}_hour'] = df[col].dt.hour       # (drift temporal)

    if SPLIT_MODE == 'temporal':
        first_dt = df.groupby('Order_Id')['order_dt'].transform('min')
        t1, t2, t3 = first_dt.quantile(list(SPLIT_Q))
        split = np.where(first_dt <= t1, 'tr', np.where(first_dt <= t2, 'va',
                         np.where(first_dt <= t3, 'vc', 'te')))
        S.dates = (str(t1)[:10], str(t2)[:10], str(t3)[:10])
    else:   # acak per Order Id (order tidak terbelah), tanpa urutan waktu
        ords = pd.Series(df['Order_Id'].unique())
        r = np.random.default_rng(0).random(len(ords))
        bucket = pd.Series(np.where(r < SPLIT_Q[0], 'tr', np.where(r < SPLIT_Q[1], 'va',
                           np.where(r < SPLIT_Q[2], 'vc', 'te'))), index=ords.values)
        split = df['Order_Id'].map(bucket).values
        S.dates = ('acak', 'acak', 'acak')
    groups = df['Order_Id'].values
    assert pd.Series(split).groupby(groups).nunique().max() == 1, 'Order terbelah antar split!'

    y = df[TARGET].astype(float).values
    period = df['order_dt'].dt.to_period('M').astype(str).values
    X = df.drop(columns=[TARGET, 'order_dt', 'ship_dt', 'order_date_DateOrders',
                         'shipping_date_DateOrders'], errors='ignore')
    for c in CAT_NUMERIC:
        if c in X.columns:
            X[c] = X[c].astype(str)
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].fillna('Unknown').astype(str).astype(object)

    idx = {s: np.where(split == s)[0] for s in ('tr', 'va', 'vc', 'te')}
    for s, v in idx.items():
        assert len(v) > 0, f'split {s} kosong'
    S.idx, S.Xall = idx, X
    S.y = {s: y[idx[s]] for s in idx}
    S.g_te = groups[idx['te']]
    S.period_te = period[idx['te']]
    S.y_min, S.y_max = float(S.y['tr'].min()), float(S.y['tr'].max())
    S.c_q = float(np.quantile(S.y['tr'], TAU))     # baseline konstanta kuantil
    S.c_mean = float(S.y['tr'].mean())             # baseline konstanta rata-rata
    print(f"   -> baris: {len(df):,} | train {len(idx['tr']):,} / val-tune {len(idx['va']):,} / val-calib {len(idx['vc']):,} / test {len(idx['te']):,}")
    print(f"   -> TARGET={TARGET} | rentang train [{S.y_min:.1f}, {S.y_max:.1f}]")
    print(f"   -> SPLIT_MODE={SPLIT_MODE} | batas: train <= {S.dates[0]} < val-tune <= {S.dates[1]} < val-calib <= {S.dates[2]} < test | tau = {TAU:.4f}")


class LevelData:
    """Subset kolom per level mitigasi + versi ordinal-encoded (fit di train) untuk LightGBM."""

    def __init__(self, level):
        self.level = level
        drop = set(LEVEL_DROP[level])
        self.cols = [c for c in S.Xall.columns if c not in drop]
        sub = S.Xall[self.cols]
        self.cat_cols = [c for c in self.cols if sub[c].dtype == object]
        self.num_cols = [c for c in self.cols if c not in self.cat_cols]
        self.raw = {s: sub.iloc[S.idx[s]].reset_index(drop=True) for s in S.idx}
        self.enc = {s: self.raw[s].copy() for s in S.idx}
        if self.cat_cols:
            oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
            oe.fit(self.raw['tr'][self.cat_cols])
            for s in S.idx:
                arr = oe.transform(self.raw[s][self.cat_cols])
                for j, c in enumerate(self.cat_cols):
                    self.enc[s][c] = arr[:, j].astype(float)


# ==============================================================================
# [2] BIAYA & METRIK
# ==============================================================================
def row_cost(y, p, co=C_OVER, cu=C_UNDER):
    d = np.asarray(p, dtype=float) - np.asarray(y, dtype=float)
    return np.where(d > 0, d * co, -d * cu)


def metrics(y, p):
    mse = mean_squared_error(y, p)
    cost = row_cost(y, p).sum()
    cq = row_cost(y, np.full(len(y), S.c_q)).sum()
    cm = row_cost(y, np.full(len(y), S.c_mean)).sum()
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    pos = y > 0
    return {'MSE': mse, 'RMSE': float(np.sqrt(mse)), 'MAE': mean_absolute_error(y, p), 'R2': r2_score(y, p),
            'MSE_minmax': mse / (S.y_max - S.y_min) ** 2, 'MAPE_pct': 100 * float(np.mean(np.abs(y[pos] - p[pos]) / y[pos])),
            'Cost': cost, 'CostRed_vsConstQ': 100 * (cq - cost) / cq, 'CostRed_vsMean': 100 * (cm - cost) / cm,
            'OverRate': float(np.mean(np.asarray(p) > np.asarray(y)))}


def rc(kind, arm):
    """Biaya per baris test dirata-ratakan antar seed (bootstrap mencerminkan variasi tuner + sampel test)."""
    return np.mean(ROW_COSTS[(kind, arm)], axis=0)


def paired_bootstrap(ca, cb, groups, n_boot, seed=0):
    """Selisih biaya total (a - b) per order; positif berarti b lebih murah."""
    d = pd.Series(ca - cb).groupby(groups).sum().values
    rng = np.random.default_rng(seed)
    n = len(d)
    boots = np.array([d[rng.integers(0, n, n)].sum() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(d.sum()), float(lo), float(hi), float((boots > 0).mean())


# ==============================================================================
# [3] MODEL LAYER (satu antarmuka: fit_predict)
# ==============================================================================
def _make_ct(num, cat):
    tr = []
    if num:
        tr.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num))
    if cat:
        tr.append(('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=30), cat))
    return ColumnTransformer(tr, sparse_threshold=1.0)


def _sub(df, idx):
    return df if idx is None else df.iloc[idx]


def _ysub(y, idx):
    return y if idx is None else y[idx]


def fit_predict(kind, ld, cols, params, tau, seed, tr_idx=None, va_idx=None, want=('va',)):
    """tau=None -> loss MSE; tau=float -> cost-aware. tr_idx/va_idx = subset posisi (proxy/subsampel)."""
    ytr, yva = _ysub(S.y['tr'], tr_idx), _ysub(S.y['va'], va_idx)
    out = {}
    if kind == 'ridge':
        num = [c for c in cols if c in ld.num_cols]
        cat = [c for c in cols if c in ld.cat_cols]
        use_cache = tr_idx is None and va_idx is None
        key = (ld.level, tuple(cols))
        if use_cache and key in RIDGE_CACHE:
            Xtr, Xva, Xte, Xvc = RIDGE_CACHE[key]
        else:
            ct = _make_ct(num, cat)
            Xtr = ct.fit_transform(_sub(ld.raw['tr'][cols], tr_idx))
            Xva = ct.transform(_sub(ld.raw['va'][cols], va_idx))
            Xte = ct.transform(ld.raw['te'][cols]) if (use_cache or 'te' in want) else None
            Xvc = ct.transform(ld.raw['vc'][cols]) if (use_cache or 'vc' in want) else None
            if use_cache:
                if len(RIDGE_CACHE) >= 6:
                    RIDGE_CACHE.pop(next(iter(RIDGE_CACHE)))
                RIDGE_CACHE[key] = (Xtr, Xva, Xte, Xvc)
        m = Ridge(alpha=params['alpha']).fit(Xtr, ytr)
        pva = m.predict(Xva)
        shift = 0.0 if tau is None else float(np.quantile(yva - pva, tau))
        out['va'] = pva + shift
        if 'te' in want:
            out['te'] = m.predict(Xte) + shift
        if 'vc' in want:
            out['vc'] = m.predict(Xvc) + shift
        out['model'] = m
        return out

    if kind == 'lgbm':
        Xtr, Xva = _sub(ld.enc['tr'][cols], tr_idx), _sub(ld.enc['va'][cols], va_idx)
        kw = dict(n_estimators=params.get('max_iter', LGB_MAX_ITER), learning_rate=params['learning_rate'],
                  num_leaves=int(params['num_leaves']), min_child_samples=int(params['min_child_samples']),
                  subsample=params['subsample'], subsample_freq=1, colsample_bytree=params['colsample_bytree'],
                  reg_lambda=params['reg_lambda'], random_state=seed, n_jobs=-1, verbose=-1)
        kw.update(dict(objective='regression') if tau is None else dict(objective='quantile', alpha=tau))
        m = lgb.LGBMRegressor(**kw)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)])
        out['va'] = m.predict(Xva)
        if 'te' in want:
            out['te'] = m.predict(ld.enc['te'][cols])
        if 'vc' in want:
            out['vc'] = m.predict(ld.enc['vc'][cols])
        out['model'] = m
        return out

    if kind == 'cat':
        cat_sel = [c for c in cols if c in ld.cat_cols]
        Xtr, Xva = _sub(ld.raw['tr'][cols], tr_idx), _sub(ld.raw['va'][cols], va_idx)
        loss = 'RMSE' if tau is None else f'Quantile:alpha={tau:.4f}'
        m = CatBoostRegressor(iterations=params.get('max_iter', CAT_MAX_ITER), learning_rate=params['learning_rate'],
                              depth=int(params['depth']), l2_leaf_reg=params['l2_leaf_reg'], loss_function=loss,
                              random_seed=seed, thread_count=-1, verbose=False, cat_features=cat_sel,
                              early_stopping_rounds=ES_ROUNDS)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
        out['va'] = m.predict(Xva)
        if 'te' in want:
            out['te'] = m.predict(ld.raw['te'][cols])
        if 'vc' in want:
            out['vc'] = m.predict(ld.raw['vc'][cols])
        out['model'] = m
        return out
    raise ValueError(kind)


def suggest_params(kind, trial):
    if kind == 'ridge':
        return {'alpha': trial.suggest_float('alpha', 1e-2, 1e3, log=True)}
    if kind == 'lgbm':
        return {'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127, log=True),
                'min_child_samples': trial.suggest_int('min_child_samples', 10, 200, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True)}
    return {'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 20.0, log=True)}


def n_trials_for(kind):
    if BUDGET_S:
        return 100 if kind == 'ridge' else 10_000      # waktu yang membatasi, bukan jumlah trial
    return min(CFG['n_trials'], 15) if kind == 'ridge' else CFG['n_trials']


def tune(kind, ld, cols, sampler_name, seed, timeout=None):
    """Optuna/Random pada VAL, objektif = rata-rata biaya asimetris. CatBoost: tuning di subsampel train."""
    n_tr = len(S.y['tr'])
    sub_idx = None
    if kind == 'cat' and CFG['cat_tune_n'] < n_tr:
        sub_idx = np.sort(np.random.default_rng(seed).choice(n_tr, CFG['cat_tune_n'], replace=False))
    nt = n_trials_for(kind)
    sampler = (optuna.samplers.TPESampler(seed=seed, n_startup_trials=max(3, nt // 4))
               if sampler_name == 'tpe' else optuna.samplers.RandomSampler(seed=seed))
    study = optuna.create_study(direction='minimize', sampler=sampler)

    def objective(trial):
        p = suggest_params(kind, trial)
        out = fit_predict(kind, ld, cols, p, TAU, seed, tr_idx=sub_idx, want=('va',))
        return float(row_cost(S.y['va'], out['va']).mean())

    t0 = time.time()
    study.optimize(objective, n_trials=nt, timeout=timeout, show_progress_bar=False)
    return study.best_params, time.time() - t0, len(study.trials)


# ==============================================================================
# [4] GWO BINER (alpha-beta-delta) UNTUK SELEKSI FITUR
# ==============================================================================
def gwo_select(kind, ld, seed):
    rng = np.random.default_rng(10_000 + seed)
    cols_all, dim = ld.cols, len(ld.cols)
    ytr, yva = S.y['tr'], S.y['va']
    tr_idx = np.sort(rng.choice(len(ytr), min(CFG['gwo_tr'], len(ytr)), replace=False))
    va_idx = np.sort(rng.choice(len(yva), min(CFG['gwo_va'], len(yva)), replace=False))
    ref = row_cost(yva[va_idx], np.full(len(va_idx), np.quantile(ytr[tr_idx], TAU))).mean()
    cache = {}

    def fix(mask):
        if mask.sum() < MIN_FEATS:
            mask[rng.choice(dim, MIN_FEATS, replace=False)] = True
        return mask

    def fitness(mask):
        key = mask.tobytes()
        if key not in cache:
            cols = [c for c, b in zip(cols_all, mask) if b]
            out = fit_predict(kind, ld, cols, PROXY[kind], TAU, seed, tr_idx, va_idx, want=('va',))
            cache[key] = row_cost(yva[va_idx], out['va']).mean() / ref + GWO_LAMBDA * mask.sum() / dim
        return cache[key]

    t0 = time.time()
    W = CFG['wolves']
    pos = rng.random((W, dim))
    masks = [fix(pos[i] > 0.5) for i in range(W)]
    fits = np.array([fitness(m) for m in masks])
    order = np.argsort(fits)[:3]
    leaders = [(fits[i], pos[i].copy(), masks[i].copy()) for i in order]      # alpha, beta, delta
    while len(leaders) < 3:
        leaders.append(leaders[-1])
    curve = [leaders[0][0]]
    for t in range(CFG['iters']):
        a = 2 - 2 * t / CFG['iters']
        new_pos = np.empty_like(pos)
        for i in range(W):
            xs = []
            for _, lp, _ in leaders:
                A = 2 * a * rng.random(dim) - a
                C = 2 * rng.random(dim)
                xs.append(lp - A * np.abs(C * lp - pos[i]))
            new_pos[i] = np.clip(np.mean(xs, axis=0), 0, 1)
        pos = new_pos
        prob = 1 / (1 + np.exp(-10 * (pos - 0.5)))
        masks = [fix(prob[i] > rng.random(dim)) for i in range(W)]
        fits = np.array([fitness(m) for m in masks])
        pool = leaders + [(fits[i], pos[i].copy(), masks[i].copy()) for i in range(W)]
        pool.sort(key=lambda z: z[0])
        leaders = pool[:3]                      # elitisme
        curve.append(leaders[0][0])
    best_mask = leaders[0][2]
    sel = [c for c, b in zip(cols_all, best_mask) if b]
    return {'cols': sel, 'curve': curve, 'n_evals': len(cache), 'seconds': time.time() - t0}


# ==============================================================================
# [5] ABLASI LEAKAGE, ENAM ARM, SENSITIVITAS
# ==============================================================================
def run_leak_ablation(LD, seed=0):
    rows, imp = [], None
    for level in ('leak', 'clean', 'strict', 'strict_nocal'):
        for kind in MODELS:
            ld = LD[level]
            t0 = time.time()
            out = fit_predict(kind, ld, ld.cols, DEFAULTS[kind], None, seed, want=('va', 'te'))
            r = {'level': level, 'model': kind, 'n_features': len(ld.cols), 'seconds': time.time() - t0}
            r.update(metrics(S.y['te'], out['te']))
            rows.append(r)
            print(f"   [{level:6s}|{kind:5s}] R2={r['R2']:.4f} RMSE={r['RMSE']:.3f} ({r['seconds']:.0f}s)")
            if kind == 'lgbm' and level == 'leak':
                imp = pd.DataFrame({'feature': ld.cols,
                                    'gain': out['model'].booster_.feature_importance(importance_type='gain')})
    return pd.DataFrame(rows), imp


def run_arms(kind, seed, ld):
    rows = []
    all_cols = ld.cols

    def evaluate(arm, cols, params, tau, extra_time, n_trials):
        t0 = time.time()
        out = fit_predict(kind, ld, cols, params, tau, seed, want=('va', 'vc', 'te'))
        r = {'model': kind, 'arm': arm, 'seed': seed, 'level': ld.level, 'n_features': len(cols),
             'n_trials': n_trials, 'seconds': time.time() - t0 + extra_time,
             'val_cost': float(row_cost(S.y['va'], out['va']).mean()), 'params': json.dumps(params, default=float)}
        r.update(metrics(S.y['te'], out['te']))
        ROW_COSTS.setdefault((kind, arm), []).append(row_cost(S.y['te'], out['te']))   # semua seed
        rows.append(r)
        PRED[(kind, seed, arm)] = {'vc': out['vc'], 'te': out['te']}
        print(f"   [{kind:5s}|{arm:15s}|seed {seed}] CostRed={r['CostRed_vsConstQ']:6.2f}% RMSE={r['RMSE']:.3f} "
              f"R2={r['R2']:.4f} nfeat={len(cols)} t={r['seconds']:.0f}s")

    evaluate('A_default_MSE', all_cols, DEFAULTS[kind], None, 0.0, 0)
    evaluate('B_default_cost', all_cols, DEFAULTS[kind], TAU, 0.0, 0)
    nt = n_trials_for(kind)
    p, dt, nd = tune(kind, ld, all_cols, 'tpe', seed, timeout=BUDGET_S)
    evaluate('C_optuna', all_cols, {**DEFAULTS[kind], **p}, TAU, dt, nd)
    p, dt, nd = tune(kind, ld, all_cols, 'random', seed, timeout=BUDGET_S)
    evaluate('F_random', all_cols, {**DEFAULTS[kind], **p}, TAU, dt, nd)
    g = gwo_select(kind, ld, seed)
    SELECTED[(kind, seed)] = g['cols']
    GWO_CURVES[(kind, seed)] = g['curve']
    print(f"      GWO: {len(g['cols'])}/{len(all_cols)} fitur, {g['n_evals']} evaluasi unik, {g['seconds']:.0f}s")
    evaluate('D_gwo', g['cols'], DEFAULTS[kind], TAU, g['seconds'], 0)
    tmo = max(30.0, BUDGET_S - g['seconds']) if BUDGET_S else None    # paritas waktu: GWO + TPE <= BUDGET_S
    p, dt, nd = tune(kind, ld, g['cols'], 'tpe', seed, timeout=tmo)
    evaluate('E_gwo_optuna', g['cols'], {**DEFAULTS[kind], **p}, TAU, g['seconds'] + dt, nd)
    return rows


def run_sensitivity(ld, seed=0):
    kind = 'lgbm'
    base = fit_predict(kind, ld, ld.cols, DEFAULTS[kind], None, seed, want=('va', 'te'))['te']
    rows = []
    for cu_r, co_r in RATIOS:
        co, cu = C_OVER, C_OVER * cu_r / co_r
        tau = cu / (cu + co)
        pq = fit_predict(kind, ld, ld.cols, DEFAULTS[kind], tau, seed, want=('va', 'te'))['te']
        yte = S.y['te']
        c_mse = row_cost(yte, base, co, cu).sum()
        c_q = row_cost(yte, pq, co, cu).sum()
        c_const = row_cost(yte, np.full(len(yte), np.quantile(S.y['tr'], tau)), co, cu).sum()
        rows.append({'c_under:c_over': f'{cu_r}:{co_r}', 'tau': tau, 'cost_mse_model': c_mse,
                     'cost_quantile_model': c_q, 'red_vs_mse_%': 100 * (c_mse - c_q) / c_mse,
                     'red_vs_constQ_%': 100 * (c_const - c_q) / c_const, 'over_rate': float(np.mean(pq > yte))})
    return pd.DataFrame(rows)


def estimate_runtime(ld):
    """Probe: ukur 1 fit penuh + 1 fit proxy per model lalu proyeksikan total (kasar)."""
    n_tr = len(S.y['tr'])
    total = 0.0
    print('[*] PROBE RUNTIME (kasar) ...')
    for kind in MODELS:
        t0 = time.time()
        fit_predict(kind, ld, ld.cols, DEFAULTS[kind], TAU, 0, want=('va', 'te'))
        t_full = time.time() - t0
        rng = np.random.default_rng(0)
        tri = np.sort(rng.choice(n_tr, min(CFG['gwo_tr'], n_tr), replace=False))
        vai = np.sort(rng.choice(len(S.y['va']), min(CFG['gwo_va'], len(S.y['va'])), replace=False))
        t0 = time.time()
        fit_predict(kind, ld, ld.cols, PROXY[kind], TAU, 0, tri, vai, want=('va',))
        t_proxy = time.time() - t0
        frac = min(1.0, CFG['cat_tune_n'] / n_tr) if kind == 'cat' else 1.0
        nt = n_trials_for(kind)
        gwo_evals = CFG['wolves'] * (CFG['iters'] + 1) * 0.85
        per_seed = 6 * t_full + 3 * nt * t_full * frac * 1.2 + gwo_evals * t_proxy
        est = 3 * t_full + 3 * t_full * 0 + len(CFG['seeds']) * per_seed
        total += est
        print(f"   {kind:5s}: fit penuh {t_full:6.1f}s | fit proxy {t_proxy:5.1f}s | proyeksi {est / 60:7.1f} menit")
    print(f"   TOTAL proyeksi (+ ablasi, sensitivitas, bootstrap ~2-5 menit): {total / 60 + 3:.0f} menit")


# ==============================================================================
# [5b] INTERVAL PREDIKSI: Persamaan 1-4 jurnal rival vs konformal (simetris / asimetris sadar-biaya)
# ==============================================================================
def eq14_interval(p_vc, y_vc, p_te, alpha=CONF_ALPHA):
    """Persamaan 1-4 Sayyad dkk.: y_hat +/- z*RMSE (RMSE dihitung pada blok kalibrasi, bukan test)."""
    from scipy.stats import norm
    z = float(norm.ppf(1 - alpha / 2))              # 1.645 untuk cakupan 90%
    rmse = float(np.sqrt(np.mean((y_vc - p_vc) ** 2)))
    return p_te - z * rmse, p_te + z * rmse


def conformal_sym(p_vc, y_vc, p_te, alpha=CONF_ALPHA):
    """Split conformal simetris: kuantil |residual| pada blok kalibrasi."""
    r = np.sort(np.abs(y_vc - p_vc))
    n = len(r)
    k = min(int(np.ceil((n + 1) * (1 - alpha))), n)
    q = r[k - 1]
    return p_te - q, p_te + q


def conformal_asym(p_vc, y_vc, p_te, alpha=CONF_ALPHA, co=C_OVER, cu=C_UNDER):
    """Cost-Asymmetric Split Conformal (CASC): miskoverage alpha dibagi menurut biaya.
    Sisi atas (under-forecast, biaya c_under tinggi) mendapat porsi miskoverage lebih kecil -> batas atas konservatif.
    Cakupan marginal >= 1 - alpha (dua sisi, union bound) bila blok kalibrasi dan test dapat dipertukarkan."""
    a_up = alpha * co / (co + cu)
    a_lo = alpha - a_up
    r = np.sort(y_vc - p_vc)
    n = len(r)
    k_up = min(int(np.ceil((n + 1) * (1 - a_up))), n)
    k_lo = max(int(np.floor((n + 1) * a_lo)), 1)
    return p_te + r[k_lo - 1], p_te + r[k_up - 1]


def interval_metrics(y, lo, hi, alpha=CONF_ALPHA):
    below, above = y < lo, y > hi
    width = hi - lo
    wink = width + (2 / alpha) * (lo - y) * below + (2 / alpha) * (y - hi) * above
    return {'coverage': float(np.mean(~below & ~above)), 'miss_low': float(below.mean()),
            'miss_high': float(above.mean()), 'mean_width': float(width.mean()),
            'width_minmax': float(width.mean() / (S.y_max - S.y_min)), 'winkler': float(wink.mean())}


INTERVAL_SPECS = [('Eq14_A', 'eq14', 'A_default_MSE'),        # replika gaya jurnal rival
                  ('Eq14_E', 'eq14', 'E_gwo_optuna'),         # Persamaan 1-4 pada model terbaik kita
                  ('SCPsym_E', 'sym', 'E_gwo_optuna'),
                  ('CASC_B', 'asym', 'B_default_cost'),
                  ('CASC_C', 'asym', 'C_optuna'),
                  ('CASC_E', 'asym', 'E_gwo_optuna')]
INTERVAL_FN = {'eq14': eq14_interval, 'sym': conformal_sym, 'asym': conformal_asym}


def run_intervals(kind, seed):
    rows, monthly = [], []
    for name, method, arm in INTERVAL_SPECS:
        pr = PRED[(kind, seed, arm)]
        lo, hi = INTERVAL_FN[method](pr['vc'], S.y['vc'], pr['te'])
        m = interval_metrics(S.y['te'], lo, hi)
        m.update({'model': kind, 'seed': seed, 'interval': name, 'arm': arm})
        rows.append(m)
        if seed == CFG['seeds'][0]:
            cov = (S.y['te'] >= lo) & (S.y['te'] <= hi)
            d = pd.DataFrame({'period': S.period_te, 'cov': cov}).groupby('period')['cov'].agg(['mean', 'size']).reset_index()
            d['model'], d['interval'] = kind, name
            monthly.append(d)
    return rows, monthly


# ==============================================================================
# [5c] REPLIKASI PROTOKOL JURNAL RIVAL (CatBoost, parameter Sayyad dkk.) + SWEEP LEARNING RATE
# ==============================================================================
def fit_rival(ld, lr, fixed=True, seed=0):
    cols = ld.cols
    kw = dict(RIVAL_PARAMS, learning_rate=lr, loss_function='RMSE', random_seed=seed, thread_count=-1,
              verbose=False, cat_features=list(ld.cat_cols))
    if not fixed:
        kw.update(iterations=1000, early_stopping_rounds=ES_ROUNDS)
    m = CatBoostRegressor(**kw)
    if fixed:
        m.fit(ld.raw['tr'][cols], S.y['tr'])
        best = kw['iterations']
    else:
        m.fit(ld.raw['tr'][cols], S.y['tr'], eval_set=(ld.raw['va'][cols], S.y['va']), use_best_model=True)
        best = int(m.get_best_iteration())
    return m.predict(ld.raw['te'][cols]), best


def run_rival_replication(LD):
    rows = []
    for level in ('leak', 'clean', 'strict'):
        ld = LD[level]
        t0 = time.time()
        p, _ = fit_rival(ld, 0.17, fixed=True)
        r = {'level': level, 'n_features': len(ld.cols), 'lr': 0.17, 'iters': RIVAL_PARAMS['iterations'],
             'seconds': time.time() - t0}
        r.update(metrics(S.y['te'], p))
        rows.append(r)
        print(f"   [rival-params|{level:6s}] MSE_minmax={r['MSE_minmax']:.3e} R2={r['R2']:.4f} "
              f"MAPE={r['MAPE_pct']:.3f}% ({r['seconds']:.0f}s)   <- jurnal rival melaporkan MSE 3.61e-07, R2 0.999")
    sweep = None
    if RUN_LR_SWEEP:
        srows = []
        for level in SWEEP_LEVELS:
            for lr in LR_SWEEP_VALUES:
                p_fix, _ = fit_rival(LD[level], lr, fixed=True)
                p_es, best = fit_rival(LD[level], lr, fixed=False)
                srows.append({'level': level, 'lr': lr,
                              'MSE_minmax_fixed300': metrics(S.y['te'], p_fix)['MSE_minmax'],
                              'MSE_minmax_earlystop': metrics(S.y['te'], p_es)['MSE_minmax'],
                              'best_iter_earlystop': best})
                print(f"   [lr-sweep|{level}] lr={lr}: fixed300 MSE_minmax={srows[-1]['MSE_minmax_fixed300']:.3e} | "
                      f"ES MSE_minmax={srows[-1]['MSE_minmax_earlystop']:.3e} (iter terbaik {best})")
        sweep = pd.DataFrame(srows)
    return pd.DataFrame(rows), sweep


def run_shift_diagnostics():
    """T8: pergeseran distribusi per bulan test (rata-rata target, porsi produk tak terlihat di train, di atas max train)."""
    te, tr = S.Xall.iloc[S.idx['te']], S.Xall.iloc[S.idx['tr']]
    d = pd.DataFrame({'period': S.period_te, 'y': S.y['te'], 'above_train_max': S.y['te'] > S.y_max})
    unseen = []
    for c in ('Product_Card_Id', 'Product_Name'):
        if c in S.Xall.columns:
            d['unseen_' + c] = (~te[c].isin(set(tr[c]))).values
            unseen.append('unseen_' + c)
    agg = dict(n=('y', 'size'), mean_y=('y', 'mean'), max_y=('y', 'max'), above_train_max=('above_train_max', 'mean'))
    agg.update({c: (c, 'mean') for c in unseen})
    g = d.groupby('period').agg(**agg).reset_index()
    g.to_csv(os.path.join(RESULTS_DIR, 'T8_shift_by_month.csv'), index=False)
    print('\n[TABEL 8] PERGESERAN DISTRIBUSI PER BULAN TEST')
    print(g.to_string(index=False, float_format=lambda v: f'{v:,.3f}'))


def seed_level_pairs(raw, pairs):
    """T3b: selisih biaya test per seed (bukti utama RQ3; bootstrap hanya mengukur sampel test)."""
    P = raw.pivot_table(index=['model', 'seed'], columns='arm', values='Cost')
    rows = []
    for kind in raw['model'].unique():
        for a, b in pairs:
            if a in P.columns and b in P.columns:
                d = (P.loc[kind][a] - P.loc[kind][b]).values            # positif = challenger lebih murah
                rows.append({'model': kind, 'baseline': a, 'challenger': b,
                             'per_seed_delta': json.dumps([round(float(x), 1) for x in d]),
                             'mean_delta': float(d.mean()), 'seeds_challenger_cheaper': f'{int((d > 0).sum())}/{len(d)}',
                             'pct_of_baseline_cost': 100 * float(d.mean()) / float(P.loc[kind][a].mean())})
    return pd.DataFrame(rows)


def selection_stability():
    """T9: frekuensi fitur terpilih GWO antar seed + Jaccard rata-rata (stabilitas seleksi, fitur kalender ditandai)."""
    from itertools import combinations
    freq, jac = [], []
    for kind in MODELS:
        sets = [set(SELECTED[(kind, sd)]) for sd in CFG['seeds'] if (kind, sd) in SELECTED]
        for f in sorted(set().union(*sets)) if sets else []:
            freq.append({'model': kind, 'feature': f, 'n_seeds': sum(f in x for x in sets), 'of': len(sets),
                         'calendar': f.startswith('order_')})
        if len(sets) > 1:
            jac.append({'model': kind, 'mean_pairwise_jaccard':
                        float(np.mean([len(x & y) / len(x | y) for x, y in combinations(sets, 2)]))})
    return pd.DataFrame(freq), pd.DataFrame(jac)


def make_interval_figure(mon):
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ref_model = MODELS[-1] if 'lgbm' not in MODELS else 'lgbm'
    for name in ('Eq14_A', 'SCPsym_E', 'CASC_E'):
        d = mon[(mon.model == ref_model) & (mon.interval == name)].sort_values('period')
        ax.plot(d['period'], d['mean'], marker='o', label=name)
    ax.axhline(1 - CONF_ALPHA, color='k', ls='--', lw=0.8, label='nominal')
    ax.set_ylabel('Cakupan empiris per bulan (test)'); ax.set_xlabel('Bulan')
    plt.xticks(rotation=60, fontsize=6); ax.legend(fontsize=7)
    ax.set_title(f'F6. Cakupan interval per bulan ({ref_model})')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'F6_interval_coverage.png')); plt.close()


# ==============================================================================
# [6] FIGUR
# ==============================================================================
def make_figures(abl, imp, raw, sens):
    plt.rcParams.update({'font.size': 9, 'figure.dpi': 200, 'savefig.dpi': 300})
    if imp is not None:
        top = imp.sort_values('gain', ascending=False).head(10)[::-1]
        fig, ax = plt.subplots(figsize=(6, 3.6))
        ax.barh(top['feature'], 100 * top['gain'] / imp['gain'].sum(), color='firebrick')
        ax.set_xlabel('Gain importance (%)')
        ax.set_title('F1. Dominasi fitur pada level L0 (LightGBM)')
        plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'F1_importance_leak.png')); plt.close()

    fig, ax = plt.subplots(figsize=(6, 3.6))
    lv = ['leak', 'clean', 'strict']
    for j, k in enumerate(MODELS):
        v = [abl[(abl.level == l) & (abl.model == k)]['R2'].values[0] for l in lv]
        ax.bar(np.arange(3) + j * 0.27, v, 0.27, label=k)
    ax.set_xticks(np.arange(3) + 0.27); ax.set_xticklabels(['L0 leak', 'L1 clean', 'L2 strict'])
    ax.set_ylabel('R2 (test)'); ax.legend(); ax.set_title('F2. Inflasi R2 akibat leakage')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'F2_leak_ablation.png')); plt.close()

    g = raw.groupby(['model', 'arm'])['CostRed_vsConstQ'].agg(['mean', 'std']).fillna(0).reset_index()
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for j, k in enumerate(MODELS):
        d = g[g.model == k].set_index('arm').reindex(ARMS)
        ax.bar(np.arange(len(ARMS)) + j * 0.27, d['mean'], 0.27, yerr=d['std'], capsize=2, label=k)
    ax.set_xticks(np.arange(len(ARMS)) + 0.27); ax.set_xticklabels([a.split('_')[0] for a in ARMS])
    ax.set_ylabel('Cost reduction vs konstanta kuantil (%)'); ax.legend(); ax.set_title('F3. Enam arm eksperimen')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'F3_arms.png')); plt.close()

    fig, ax = plt.subplots(figsize=(6, 3.6))
    for (k, sd), c in GWO_CURVES.items():
        ax.plot(range(len(c)), c, marker='o', label=f'{k} s{sd}')
    ax.set_xlabel('Iterasi GWO'); ax.set_ylabel('Fitness terbaik'); ax.legend(fontsize=6)
    ax.set_title('F4. Konvergensi GWO')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'F4_gwo_convergence.png')); plt.close()

    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(sens['c_under:c_over'], sens['red_vs_mse_%'], marker='o', label='vs model MSE')
    ax.plot(sens['c_under:c_over'], sens['red_vs_constQ_%'], marker='s', label='vs konstanta kuantil')
    ax.set_xlabel('Rasio c_under : c_over'); ax.set_ylabel('Penghematan biaya (%)'); ax.legend()
    ax.set_title('F5. Sensitivitas rasio biaya (LightGBM)')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'F5_sensitivity.png')); plt.close()


# ==============================================================================
# [7] MAIN
# ==============================================================================
def main():
    t_start = time.time()
    print(f"[*] MODE={MODE} | MAIN_LEVEL={MAIN_LEVEL} | MODELS={MODELS} | seeds={CFG['seeds']} | "
          f"n_trials={CFG['n_trials']} | GWO={CFG['wolves']}x{CFG['iters']}")
    load_dataset(CFG)
    LD = {lv: LevelData(lv) for lv in ('leak', 'clean', 'strict', 'strict_nocal')}
    for lv, ld in LD.items():
        print(f"   level {lv:6s}: {len(ld.cols)} fitur ({len(ld.num_cols)} num, {len(ld.cat_cols)} kat)")
    ld_main = LD[MAIN_LEVEL]
    run_shift_diagnostics()
    if RUN_PROBE and not ONLY_TABLE1:
        estimate_runtime(ld_main)

    print('\n[TABEL 1] ABLASI LEAKAGE (default, loss MSE)')
    abl, imp = run_leak_ablation(LD)
    abl.to_csv(os.path.join(RESULTS_DIR, 'T1_leakage_ablation.csv'), index=False)
    if RUN_RIVAL:
        print('\n[TABEL 7] REPLIKASI PROTOKOL RIVAL (CatBoost, parameter Sayyad dkk.)')
        rv, sw = run_rival_replication(LD)
        rv.to_csv(os.path.join(RESULTS_DIR, 'T7a_rival_replication.csv'), index=False)
        if sw is not None:
            sw.to_csv(os.path.join(RESULTS_DIR, 'T7b_lr_sweep.csv'), index=False)
    if ONLY_TABLE1:
        print(f'\n[OK] ONLY_TABLE1 selesai. Keluaran: {RESULTS_DIR}')
        return

    print(f'\n[TABEL 2] ENAM ARM pada level {MAIN_LEVEL}')
    raw_path = os.path.join(RESULTS_DIR, 'T2_arms_raw.csv')
    all_rows, int_rows, mon_rows = [], [], []
    for kind in MODELS:
        for seed in CFG['seeds']:
            all_rows += run_arms(kind, seed, ld_main)
            ir, im = run_intervals(kind, seed)
            int_rows += ir
            mon_rows += im
            pd.DataFrame(all_rows).to_csv(raw_path, index=False)      # checkpoint
            pd.DataFrame(int_rows).to_csv(os.path.join(RESULTS_DIR, 'T5_intervals_raw.csv'), index=False)
    raw = pd.DataFrame(all_rows)
    agg = raw.groupby(['model', 'arm']).agg(
        CostRed_mean=('CostRed_vsConstQ', 'mean'), CostRed_std=('CostRed_vsConstQ', 'std'),
        CostRed_vsMean=('CostRed_vsMean', 'mean'), RMSE=('RMSE', 'mean'), R2=('R2', 'mean'),
        OverRate=('OverRate', 'mean'), n_features=('n_features', 'mean'), seconds=('seconds', 'mean')
    ).fillna(0).reset_index()
    agg['arm'] = pd.Categorical(agg['arm'], ARMS, ordered=True)
    agg = agg.sort_values(['model', 'arm'])
    agg.to_csv(os.path.join(RESULTS_DIR, 'T2_arms_aggregated.csv'), index=False)
    print(agg.to_string(index=False, float_format=lambda v: f'{v:,.3f}'))

    print('\n[TABEL 3] PAIRED BOOTSTRAP TINGKAT-ORDER (biaya dirata-ratakan antar seed)')
    pairs = [('A_default_MSE', 'B_default_cost'), ('B_default_cost', 'C_optuna'), ('B_default_cost', 'D_gwo'),
             ('B_default_cost', 'E_gwo_optuna'), ('C_optuna', 'E_gwo_optuna'), ('F_random', 'C_optuna'),
             ('F_random', 'E_gwo_optuna')]
    brow = []
    for kind in MODELS:
        for a, b in pairs:
            if (kind, a) in ROW_COSTS and (kind, b) in ROW_COSTS:
                d, lo, hi, pb = paired_bootstrap(rc(kind, a), rc(kind, b), S.g_te, CFG['boot'])
                brow.append({'model': kind, 'baseline': a, 'challenger': b, 'delta_cost(a-b)': d,
                             'ci95_lo': lo, 'ci95_hi': hi, 'P(challenger cheaper)': pb,
                             'significant': bool(lo > 0 or hi < 0)})
    for arm in ('B_default_cost', 'E_gwo_optuna'):          # RQ2: antar-learner pada arm yang sama
        for a, b in (('lgbm', 'cat'), ('ridge', 'cat'), ('ridge', 'lgbm')):
            if (a, arm) in ROW_COSTS and (b, arm) in ROW_COSTS:
                d, lo, hi, pb = paired_bootstrap(rc(a, arm), rc(b, arm), S.g_te, CFG['boot'])
                brow.append({'model': f'{a}_vs_{b}', 'baseline': f'{a}:{arm}', 'challenger': f'{b}:{arm}',
                             'delta_cost(a-b)': d, 'ci95_lo': lo, 'ci95_hi': hi, 'P(challenger cheaper)': pb,
                             'significant': bool(lo > 0 or hi < 0)})
    boot = pd.DataFrame(brow)
    boot.to_csv(os.path.join(RESULTS_DIR, 'T3_bootstrap.csv'), index=False)
    t3b = seed_level_pairs(raw, pairs)
    t3b.to_csv(os.path.join(RESULTS_DIR, 'T3b_seedlevel_pairs.csv'), index=False)
    print('\n[TABEL 3b] SELISIH BIAYA PER SEED (bukti utama RQ3)')
    print(t3b.to_string(index=False, float_format=lambda v: f'{v:,.2f}'))
    print(boot.to_string(index=False, float_format=lambda v: f'{v:,.3f}'))

    print('\n[TABEL 4] SENSITIVITAS RASIO BIAYA (LightGBM default)')
    sens = run_sensitivity(ld_main)
    sens.to_csv(os.path.join(RESULTS_DIR, 'T4_sensitivity.csv'), index=False)
    print(sens.to_string(index=False, float_format=lambda v: f'{v:,.3f}'))

    print('\n[TABEL 5] INTERVAL PREDIKSI (cakupan nominal %d%%)' % round(100 * (1 - CONF_ALPHA)))
    iraw = pd.DataFrame(int_rows)
    iagg = iraw.groupby(['model', 'interval']).agg(
        coverage=('coverage', 'mean'), coverage_std=('coverage', 'std'), miss_low=('miss_low', 'mean'),
        miss_high=('miss_high', 'mean'), width_minmax=('width_minmax', 'mean'), winkler=('winkler', 'mean')
    ).fillna(0).reset_index()
    iagg.to_csv(os.path.join(RESULTS_DIR, 'T5_intervals_aggregated.csv'), index=False)
    print(iagg.to_string(index=False, float_format=lambda v: f'{v:,.4f}'))
    mon = pd.concat(mon_rows, ignore_index=True)
    mon.to_csv(os.path.join(RESULTS_DIR, 'T6_coverage_by_month.csv'), index=False)
    make_interval_figure(mon)

    t9a, t9b = selection_stability()
    t9a.to_csv(os.path.join(RESULTS_DIR, 'T9a_gwo_feature_frequency.csv'), index=False)
    t9b.to_csv(os.path.join(RESULTS_DIR, 'T9b_gwo_jaccard.csv'), index=False)
    print('\n[TABEL 9] STABILITAS SELEKSI FITUR GWO (Jaccard antar seed)')
    print(t9b.to_string(index=False, float_format=lambda v: f'{v:,.3f}'))

    with open(os.path.join(RESULTS_DIR, 'selected_features_and_config.json'), 'w') as f:
        json.dump({'config': {'MODE': MODE, 'MAIN_LEVEL': MAIN_LEVEL, 'C_OVER': C_OVER, 'C_UNDER': C_UNDER,
                              'TAU': TAU, 'SPLIT_MODE': SPLIT_MODE, 'TARGET': TARGET, 'BUDGET_S': BUDGET_S, 'CONF_ALPHA': CONF_ALPHA, 'split_dates': S.dates, **CFG},
                   'selected': {f'{k}_seed{s}': v for (k, s), v in SELECTED.items()}}, f, indent=2, default=str)
    make_figures(abl, imp, raw, sens)
    print(f"\n[OK] Selesai dalam {(time.time() - t_start) / 60:.1f} menit. Keluaran: {RESULTS_DIR}")


if __name__ == '__main__' and RUN_PIPELINE:
    main()

In [ ]:
import os
import numpy as np
import pandas as pd

RAW_CSV_PATH = os.environ.get('DATACO_CSV', '/content/drive/MyDrive/DataCoSupplyChainDataset.csv')
df = pd.read_csv(RAW_CSV_PATH, encoding='latin1')

y   = df["Sales per customer"]
tot = df["Order Item Total"]
rec = df["Order Item Quantity"] * df["Order Item Product Price"] * (1 - df["Order Item Discount Rate"])

d1 = (y - tot).abs()
d2 = (y - rec).abs()

print("max |SPC - Order Item Total|      =", d1.max())
print("max |SPC - qty*price*(1-disc)|    =", d2.max())
print("rows with diff > 1e-6:", (d1 > 1e-6).sum(), (d2 > 1e-6).sum(), "of", len(df))